# Driver Fatigue Classification and Progression

This notebook implements the requested image pipeline using the supplied four-folder dataset. It trains and compares a custom CNN and a frozen ImageNet MobileNetV2 model, trains separate eye and mouth classifiers, evaluates held-out data, fuses four-class predictions into three fatigue stages, and plots interval-level progression.

**Important data limits:** the supplied ZIP has 2,900 images under one `train/` folder. This notebook makes a stratified 70/15/15 image-level split. If multiple images come from the same person or recording, they may cross splits; a subject/video-level split is preferable when IDs are available. No dataset citation or license is included in the ZIP, so verify reuse rights before publication. The progression demonstration requires real, time-ordered session frames; a shuffled test-set demo is explicitly labeled synthetic. Model outputs are research/education aids, not a driver safety device.


## 1. Setup and dataset extraction

Place the supplied ZIP next to this notebook, or set `ZIP_PATH` below. The archive is extracted into `data/`.

In [ ]:
from pathlib import Path
import zipfile, time, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, precision_recall_fscore_support)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 12  # increase if needed; EarlyStopping restores best validation weights
CLASS_NAMES = ["Closed", "Open", "no_yawn", "yawn"]

ZIP_CANDIDATES = [Path("archive (2) (1)(1).zip"), Path("archive (2) (1).zip")]
ZIP_PATH = next((p for p in ZIP_CANDIDATES if p.exists()), ZIP_CANDIDATES[0])
DATA_DIR = Path("data")
if ZIP_PATH.exists():
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(DATA_DIR)
else:
    print(f"ZIP not found at {ZIP_PATH}. Set ZIP_PATH to the uploaded archive location.")

DATA_ROOT = DATA_DIR / "train"
if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Expected class folders under {DATA_ROOT}; update DATA_ROOT for your extracted dataset.")
print("TensorFlow:", tf.__version__)
print("Data root:", DATA_ROOT.resolve())


## 2. Inspect labels and create leakage-aware splits

The available archive is almost balanced. The split below is stratified by class and reproducible. It cannot prevent the same person/session appearing in more than one split because identity/session metadata is absent.

In [ ]:
extensions = {".jpg", ".jpeg", ".png", ".bmp"}
records = [(p, p.parent.name) for p in DATA_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in extensions]
counts = pd.Series([label for _, label in records]).value_counts().reindex(CLASS_NAMES, fill_value=0)
display(counts.rename("images").to_frame())
assert set(counts.index[counts > 0]) == set(CLASS_NAMES), f"Expected classes {CLASS_NAMES}; found {counts[counts>0].index.tolist()}"
paths = np.array([str(p) for p, _ in records])
labels = np.array([CLASS_NAMES.index(label) for _, label in records], dtype=np.int32)
X_train, X_temp, y_train, y_temp = train_test_split(paths, labels, test_size=0.30, stratify=labels, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)
print(f"Train={len(X_train)}, validation={len(X_val)}, test={len(X_test)}")


## 3. Decode, resize, normalize, and augment

Images are resized to 224×224 and scaled to 0–1. Augmentation is applied only during training. The augmentations are intentionally modest; inspect samples and tune them for your camera domain.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
def decode_resize(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def make_dataset(paths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training: ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    return ds.map(decode_resize, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
train_ds = make_dataset(X_train, y_train, True)
val_ds = make_dataset(X_val, y_val)
test_ds = make_dataset(X_test, y_test)

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomBrightness(0.15, value_range=(0, 1)),
], name="train_augmentation")


## 4. Model builders: custom CNN and MobileNetV2

Both models produce four classes for a fair comparison. MobileNetV2's convolutional base is frozen; the task-specific head is trained with Adam and sparse categorical cross-entropy. ImageNet weights are downloaded by Keras on first use if they are not cached.

In [ ]:
def build_custom_cnn(num_classes=4):
    inputs = tf.keras.Input((*IMG_SIZE, 3))
    x = augment(inputs)
    for filters in (32, 64, 128, 256):
        x = tf.keras.layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.MaxPooling2D()(x)
        x = tf.keras.layers.Dropout(0.15)(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.35)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs, name="custom_cnn")

def build_mobilenet(num_classes=4):
    inputs = tf.keras.Input((*IMG_SIZE, 3))
    x = augment(inputs)
    # Dataset tensors are [0,1]; MobileNetV2 expects [-1,1].
    x = tf.keras.layers.Rescaling(2.0, offset=-1.0)(x)
    base = tf.keras.applications.MobileNetV2(include_top=False, weights="imagenet", input_shape=(*IMG_SIZE, 3))
    base.trainable = False
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.30)(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.25)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs, name="mobilenetv2_transfer")
    return model

def compile_model(model):
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6),
]


## 5. Train and compare the four-class models

Training time is measured on the current runtime and depends on CPU/GPU, batch size, and TensorFlow version. The test set is used only after training. Models and training histories are saved under `artifacts/`.

In [ ]:
ARTIFACTS = Path("artifacts"); ARTIFACTS.mkdir(exist_ok=True)
def fit_and_time(model, name):
    compile_model(model)
    start = time.perf_counter()
    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks, verbose=1)
    elapsed = time.perf_counter() - start
    model.save(ARTIFACTS / f"{name}.keras")
    return model, history, elapsed

custom_model, custom_history, custom_seconds = fit_and_time(build_custom_cnn(), "custom_cnn_4class")
transfer_model, transfer_history, transfer_seconds = fit_and_time(build_mobilenet(), "mobilenetv2_4class")


## 6. Evaluation on unseen image-level test split

Accuracy, per-class precision/recall/F1, confusion matrices, and train–validation accuracy gap are reported. A lower score on `Closed` or `yawn` can have different safety consequences than errors on the other classes, so inspect each row.

In [ ]:
def evaluate_model(model, name):
    probs = model.predict(test_ds, verbose=0)
    pred = probs.argmax(axis=1)
    print(f"\n{name} test accuracy: {accuracy_score(y_test, pred):.4f}")
    print(classification_report(y_test, pred, labels=range(4), target_names=CLASS_NAMES, zero_division=0))
    cm = confusion_matrix(y_test, pred, labels=range(4))
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(cmap="Blues", xticks_rotation=35)
    plt.title(f"{name} — test confusion matrix"); plt.tight_layout(); plt.show()
    return probs, pred, cm

custom_probs, custom_pred, custom_cm = evaluate_model(custom_model, "Custom CNN")
transfer_probs, transfer_pred, transfer_cm = evaluate_model(transfer_model, "MobileNetV2")

def best_val_accuracy(history): return float(max(history.history.get("val_accuracy", [np.nan])))
def test_accuracy(pred): return float(accuracy_score(y_test, pred))
comparison = pd.DataFrame([
    {"model":"Custom CNN", "train_seconds":custom_seconds, "best_val_accuracy":best_val_accuracy(custom_history), "test_accuracy":test_accuracy(custom_pred), "val_test_gap":best_val_accuracy(custom_history)-test_accuracy(custom_pred)},
    {"model":"MobileNetV2 frozen", "train_seconds":transfer_seconds, "best_val_accuracy":best_val_accuracy(transfer_history), "test_accuracy":test_accuracy(transfer_pred), "val_test_gap":best_val_accuracy(transfer_history)-test_accuracy(transfer_pred)},
])
display(comparison)
comparison.to_csv(ARTIFACTS / "model_comparison.csv", index=False)


## 7. Separate eye-state and mouth-state classifiers

Two binary MobileNetV2 heads are trained independently: `Closed/Open` and `no_yawn/yawn`. We retain each model's class ordering explicitly for correct interpretation.

In [ ]:
def fit_and_time_on(model, training_data, validation_data, name):
    compile_model(model); start=time.perf_counter()
    history=model.fit(training_data,validation_data=validation_data,epochs=EPOCHS,callbacks=callbacks,verbose=1)
    elapsed=time.perf_counter()-start; model.save(ARTIFACTS/f"{name}.keras")
    return model,history,elapsed


In [ ]:
EYE_CLASS_NAMES = ["Closed", "Open"]
MOUTH_CLASS_NAMES = ["no_yawn", "yawn"]
def binary_split(class_pair):
    keep = np.isin(labels, [CLASS_NAMES.index(c) for c in class_pair])
    p, y = paths[keep], np.array([class_pair.index(CLASS_NAMES[i]) for i in labels[keep]], dtype=np.int32)
    p_train,p_tmp,y_train_b,y_tmp=train_test_split(p,y,test_size=.30,stratify=y,random_state=SEED)
    p_val,p_test,y_val_b,y_test_b=train_test_split(p_tmp,y_tmp,test_size=.50,stratify=y_tmp,random_state=SEED)
    return (make_dataset(p_train,y_train_b,True), make_dataset(p_val,y_val_b), make_dataset(p_test,y_test_b), y_test_b)

def build_binary_mobilenet():
    return build_mobilenet(num_classes=2)

eye_train,eye_val,eye_test_ds,y_eye_test=binary_split(EYE_CLASS_NAMES)
mouth_train,mouth_val,mouth_test_ds,y_mouth_test=binary_split(MOUTH_CLASS_NAMES)
eye_model,eye_history,eye_seconds=fit_and_time_on(build_binary_mobilenet(), eye_train, eye_val, "mobilenetv2_eye")
mouth_model,mouth_history,mouth_seconds=fit_and_time_on(build_binary_mobilenet(), mouth_train, mouth_val, "mobilenetv2_mouth")


In [ ]:
def binary_evaluation(model, test_ds_b, y_true, names, title):
    probs=model.predict(test_ds_b,verbose=0); pred=probs.argmax(axis=1)
    print(title, "accuracy:", round(accuracy_score(y_true,pred),4))
    print(classification_report(y_true,pred,labels=range(2),target_names=names,zero_division=0))
    ConfusionMatrixDisplay(confusion_matrix(y_true,pred,labels=range(2)),display_labels=names).plot(cmap="Blues")
    plt.title(title); plt.tight_layout(); plt.show()
    return probs,pred
eye_probs,eye_pred=binary_evaluation(eye_model,eye_test_ds,y_eye_test,EYE_CLASS_NAMES,"Eye state")
mouth_probs,mouth_pred=binary_evaluation(mouth_model,mouth_test_ds,y_mouth_test,MOUTH_CLASS_NAMES,"Mouth state")


## 8. Four-class to three-stage fatigue fusion

The requested rule is applied to the four-class model's top prediction: `Open` or `no_yawn` → Alert (0), `yawn` → Mild Fatigue (1), and `Closed` → Severe Fatigue (2). This hard rule can miss simultaneous signs (for example, a yawn while eyes are closed). For deployment, consider calibrated probabilities and temporal persistence rules.

In [ ]:
FATIGUE_NAMES = ["Alert", "Mild Fatigue", "Severe Fatigue"]
CLASS_TO_STAGE = {"Open":0, "no_yawn":0, "yawn":1, "Closed":2}
def four_class_to_stage(class_predictions):
    return np.array([CLASS_TO_STAGE[CLASS_NAMES[int(i)]] for i in class_predictions], dtype=int)

test_stages=four_class_to_stage(transfer_pred)
print("Four-class test predictions mapped to fatigue stages:")
print(pd.Series([FATIGUE_NAMES[i] for i in test_stages]).value_counts())


## 9. Sequential-session progression curve

Set `SESSION_DIR` to a folder containing chronological frames (filenames should sort in capture order). Set the number of frames corresponding to one minute for your sampling rate. If no session folder is supplied, the notebook uses shuffled held-out images only to illustrate the code path; this is synthetic and must not be interpreted as a real driver's progression.

In [ ]:
SESSION_DIR = Path("session_frames")  # add real, time-ordered frames here
FRAMES_PER_MINUTE = 60  # example: 1 frame/second; adjust to actual capture rate
if SESSION_DIR.exists():
    session_paths=sorted([p for p in SESSION_DIR.iterdir() if p.suffix.lower() in extensions])
    session_is_synthetic=False
else:
    rng=np.random.default_rng(SEED)
    session_paths=[Path(p) for p in rng.permutation(X_test)[:min(300,len(X_test))]]
    session_is_synthetic=True
if not session_paths: raise ValueError("No session frames found.")

def load_one(path):
    raw=tf.io.read_file(str(path)); img=tf.io.decode_image(raw,channels=3,expand_animations=False)
    img=tf.image.resize(img,IMG_SIZE); return tf.cast(img,tf.float32)/255.0
frame_batch=tf.stack([load_one(p) for p in session_paths])
frame_probs=transfer_model.predict(frame_batch,batch_size=BATCH_SIZE,verbose=0)
frame_class=frame_probs.argmax(axis=1)
frame_stage=four_class_to_stage(frame_class)
rows=[]
for i in range(0,len(frame_stage),FRAMES_PER_MINUTE):
    stages=frame_stage[i:i+FRAMES_PER_MINUTE]
    mean_stage=float(stages.mean())
    interval_stage=0 if mean_stage < .5 else (1 if mean_stage < 1.5 else 2)
    rows.append({"minute_interval":i//FRAMES_PER_MINUTE+1,"mean_stage_score":mean_stage,
                 "interval_stage":interval_stage,"stage_name":FATIGUE_NAMES[interval_stage],
                 "alert_fraction":float(np.mean(stages==0)),"mild_fraction":float(np.mean(stages==1)),
                 "severe_fraction":float(np.mean(stages==2))})
progression=pd.DataFrame(rows)
display(progression)
progression.to_csv(ARTIFACTS/"fatigue_progression.csv",index=False)
plt.figure(figsize=(10,4)); plt.plot(progression.minute_interval,progression.mean_stage_score,marker="o")
plt.axhline(.5,color="orange",linestyle="--",label="Alert → Mild threshold")
plt.axhline(1.5,color="red",linestyle="--",label="Mild → Severe threshold")
plt.yticks([0,1,2],FATIGUE_NAMES); plt.xlabel("Minute interval"); plt.ylabel("Mean predicted fatigue stage")
plt.title("Fatigue progression" + (" (synthetic shuffled demo)" if session_is_synthetic else " (session frames)"))
plt.grid(alpha=.25); plt.legend(); plt.tight_layout(); plt.show()

def first_interval_of(stage_id):
    matches=progression.index[progression.interval_stage==stage_id]
    return None if len(matches)==0 else int(progression.loc[matches[0],"minute_interval"])
print("Session type:", "SYNTHETIC — do not interpret as a real session" if session_is_synthetic else "user-provided chronological frames")
print("First interval labeled Mild Fatigue:", first_interval_of(1))
print("First interval labeled Severe Fatigue:", first_interval_of(2))


## 10. Robustness checks and limitations

The next cell evaluates the transfer model on a darker and a brighter version of the held-out set. This is a simple stress test, not a substitute for testing real night driving, sunglasses, occlusion, camera placement, demographics, or different vehicles. Report performance by class and examine false-alert/false-safe cases.

In [ ]:
def brightness_stress(dataset, delta):
    y_true=[]; y_pred=[]
    for images,labels_b in dataset:
        stressed=tf.clip_by_value(images+delta,0.,1.)
        y_pred.extend(transfer_model.predict(stressed,verbose=0).argmax(axis=1))
        y_true.extend(labels_b.numpy())
    return accuracy_score(y_true,y_pred)
print(pd.DataFrame([
    {"condition":"original test", "accuracy":test_accuracy(transfer_pred)},
    {"condition":"darker (brightness -0.15)", "accuracy":brightness_stress(test_ds,-0.15)},
    {"condition":"brighter (brightness +0.15)", "accuracy":brightness_stress(test_ds,0.15)},
]))


## 11. Interpretation and reporting checklist

When run, use the generated tables and plots to complete a report with actual measured values. Do not claim high performance unless the held-out metrics support it.

- Compare test accuracy, per-class precision/recall, confusion matrices, training time, and validation–test gap for custom CNN vs. MobileNetV2.
- Check whether Closed and Yawn are confused with other classes; these are key safety errors.
- Confirm image provenance, license, and whether splits are separated by person/video.
- Collect a distinct, real chronological driving session for progression analysis.
- Document lighting, pose, occlusion, camera, and demographic limitations.
- Treat the model as a prototype; do not rely on it for real-time driver safety decisions without independent validation.

The run artifacts (`*.keras`, `model_comparison.csv`, and `fatigue_progression.csv`) are saved under `artifacts/`.